In [ ]:
import pandas as pd
import numpy as np

PRICE_FILE = "1_data_.xlsx"
EARNINGS_FILE = "2_data_.xlsx"
RFM_FILE = "RFM-11-sector.csv"

SCREEN_DATE = "2026-03-25"
TOP_N = 9
MCAP_MIN = 500_000
TURNOVER_MIN = 5_000_000_000


def clean_code(x):
    return str(x).strip() if pd.notna(x) else np.nan


def get_date_cols(df, exclude_cols):
    date_cols = []
    for col in df.columns:
        if col in exclude_cols:
            continue
        try:
            pd.to_datetime(col, errors="raise")
            date_cols.append(col)
        except Exception:
            continue
    return date_cols


def melt_fnguide(df, keyword, value_name, date_col="date"):
    subset = df[df["아이템명"].astype(str).str.contains(keyword, na=False)].copy()

    id_cols = [c for c in ["코드", "코드명", "아이템명"] if c in subset.columns]
    date_cols = get_date_cols(subset, id_cols)

    subset = subset[id_cols + date_cols].copy()
    subset["코드"] = subset["코드"].map(clean_code)

    out = subset.melt(
        id_vars=id_cols,
        value_vars=date_cols,
        var_name=date_col,
        value_name=value_name
    )

    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    out = out.dropna(subset=[date_col])

    return out


price_raw = pd.read_excel(PRICE_FILE, header=8)
earn_raw = pd.read_excel(EARNINGS_FILE, header=8)
rfm = pd.read_csv(RFM_FILE)

rfm_codes = set(rfm["종목코드"].astype(str).str.strip())

price = melt_fnguide(price_raw, "수정주가", "price")
mcap = melt_fnguide(price_raw, "시가총액", "mcap")
turn = melt_fnguide(price_raw, "거래대금", "turnover")

daily = price.merge(mcap[["코드", "date", "mcap"]], on=["코드", "date"], how="left")
daily = daily.merge(turn[["코드", "date", "turnover"]], on=["코드", "date"], how="left")

daily["코드"] = daily["코드"].map(clean_code)
daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily = daily.dropna(subset=["코드", "date"])
daily = daily[daily["코드"].isin(rfm_codes)].copy()
daily = daily.sort_values(["코드", "date"]).reset_index(drop=True)

daily["ret_1m"] = daily.groupby("코드")["price"].pct_change(20)
daily["ret_3m"] = daily.groupby("코드")["price"].pct_change(60)

earn = melt_fnguide(earn_raw, "영업이익", "op_income", "quarter")
earn["코드"] = earn["코드"].map(clean_code)
earn["quarter"] = pd.to_datetime(earn["quarter"], errors="coerce")
earn = earn.dropna(subset=["코드", "quarter"]).copy()
earn = earn.sort_values(["코드", "quarter"]).reset_index(drop=True)

earn["op_lag4"] = earn.groupby("코드")["op_income"].shift(4)
earn["op_yoy"] = (earn["op_income"] - earn["op_lag4"]) / earn["op_lag4"].abs()
earn["op_yoy"] = earn["op_yoy"].replace([np.inf, -np.inf], np.nan).clip(-5, 5)
earn = earn[["코드", "quarter", "op_yoy"]].drop_duplicates(subset=["코드", "quarter"], keep="last")

merged = []
for code, g in daily.groupby("코드", sort=False):
    e = earn[earn["코드"] == code].sort_values("quarter").copy()
    g = g.sort_values("date").reset_index(drop=True)

    if e.empty:
        g["op_yoy"] = np.nan
    else:
        g = pd.merge_asof(
            g,
            e,
            left_on="date",
            right_on="quarter",
            direction="backward",
            allow_exact_matches=True
        )
    merged.append(g)

daily = pd.concat(merged, ignore_index=True)
daily = daily.sort_values(["코드", "date"]).reset_index(drop=True)

screen = daily[daily["date"] == pd.to_datetime(SCREEN_DATE)].copy()
screen = screen[
    (screen["mcap"] >= MCAP_MIN) &
    (screen["turnover"] >= TURNOVER_MIN)
].copy()

screen["rank_1m"] = screen["ret_1m"].rank(ascending=False, method="min")
screen["rank_3m"] = screen["ret_3m"].rank(ascending=False, method="min")
screen["rank_op"] = screen["op_yoy"].rank(ascending=False, method="min")

screen = screen.dropna(subset=["rank_1m", "rank_3m", "rank_op"]).copy()
screen["score"] = screen["rank_1m"] + screen["rank_3m"] + screen["rank_op"]

screened_portfolio = screen.sort_values(
    ["score", "rank_1m", "rank_3m", "rank_op", "코드"]
).head(TOP_N)

screened_portfolio = screened_portfolio[
    [
        "코드", "코드명", "price", "mcap", "turnover",
        "ret_1m", "ret_3m", "op_yoy",
        "rank_1m", "rank_3m", "rank_op", "score"
    ]
].reset_index(drop=True)

screened_portfolio.to_csv("screened_portfolio.csv", index=False, encoding="utf-8-sig")
screened_portfolio